# Algoritma Genetika: Studi Kasus Word Matching

In [13]:
import random
import numpy as np
import matplotlib.pyplot as plt

Target_word = "GENETIKA"
target = [ord(c) - ord('A') for c in Target_word]

print("Target word:", Target_word)
print("Target word (numerical):", target)
print("panjang target:", len(target))

Target word: GENETIKA
Target word (numerical): [6, 4, 13, 4, 19, 8, 10, 0]
panjang target: 8


# Parameter Algoritma Genetika

In [14]:
jumlah_populasi = 100
jumlah_generasi = 10
prob_crossover = 0.7
prob_mutasi = 0.01
panjang_gen = len(target)
min_allele = 1
max_allele = 26
max_fitness = max_allele * panjang_gen

print("Parameter Algoritma Genetika:")
print("Jumlah populasi:", jumlah_populasi)
print("Jumlah generasi:", jumlah_generasi)
print("Probabilitas crossover:", prob_crossover)
print("Probabilitas mutasi:", prob_mutasi)
print("Panjang gen:", panjang_gen)
print("Rentang allele:", min_allele, "sampai", max_allele)
print("Fitness maksimal:", max_fitness)

Parameter Algoritma Genetika:
Jumlah populasi: 100
Jumlah generasi: 10
Probabilitas crossover: 0.7
Probabilitas mutasi: 0.01
Panjang gen: 8
Rentang allele: 1 sampai 26
Fitness maksimal: 208


# Fungsi Helper

In [15]:
def angka_ke_huruf(individu):
    """
    Mengubah list angka menjadi string kata.
    Contoh: [7,5,14,5,20,9,11,1] -> 'GENETIKA'
    """
    return ''.join([chr(g + ord('A') - 1) for g in individu])

# Uji coba fungsi
print("Uji angka_ke_huruf:")
print(f"  {target} -> '{angka_ke_huruf(target)}'")
print(f"  [1,2,3] -> '{angka_ke_huruf([1,2,3])}'")

Uji angka_ke_huruf:
  [6, 4, 13, 4, 19, 8, 10, 0] -> 'FDMDSHJ@'
  [1,2,3] -> 'ABC'


# Pembangkit Populasi Awal

In [16]:
def create_individual():
    return [random.randint(min_allele, max_allele) for _ in range(panjang_gen)]

def create_population(jumlah_populasi):
    return [create_individual() for _ in range(jumlah_populasi)]

populasi_example = create_population(3)
print("Contoh individu dalam populasi:")

for i, individu in enumerate(populasi_example):
    print(f'Individu {i+1}: {individu} -> {angka_ke_huruf(individu)}', individu)

Contoh individu dalam populasi:
Individu 1: [18, 7, 12, 22, 23, 5, 19, 16] -> RGLVWESP [18, 7, 12, 22, 23, 5, 19, 16]
Individu 2: [9, 13, 11, 26, 13, 18, 26, 4] -> IMKZMRZD [9, 13, 11, 26, 13, 18, 26, 4]
Individu 3: [4, 12, 3, 8, 13, 23, 5, 17] -> DLCHMWEQ [4, 12, 3, 8, 13, 23, 5, 17]


# Fungsi Fitness

In [17]:
def calculate_fitness(individu,target):
    total_selisih = sum(abs(individu[i] - target[i]) for i in range(len(target)))
    fitness = max_fitness - total_selisih
    return fitness

def calculate_fitness_all(population, target):
    return [calculate_fitness(individu, target) for individu in population]

individu_example = create_individual()
fitness_example = calculate_fitness(individu_example, target)
print("Uji Fungsi Fitness (contoh dari slide):")
print(f"  Individu : {individu_example} = '{angka_ke_huruf(individu_example)}'")
print(f"  Target   : {target} = '{angka_ke_huruf(target)}'")
print(f"  Fitness  : {max_fitness} - {sum(abs(individu_example[i] - target[i]) for i in range(len(target)))} = {fitness_example}  (harusnya 162)")
print()
print(f"  Fitness TARGET itu sendiri: {calculate_fitness(target, target)}  (harusnya {max_fitness})")

Uji Fungsi Fitness (contoh dari slide):
  Individu : [24, 1, 19, 14, 23, 7, 21, 4] = 'XASNWGUD'
  Target   : [6, 4, 13, 4, 19, 8, 10, 0] = 'FDMDSHJ@'
  Fitness  : 208 - 57 = 151  (harusnya 162)

  Fitness TARGET itu sendiri: 208  (harusnya 208)


# Fungsi Seleksi (Roullete Wheel)

In [18]:
def select_roullette(population, fitness_values):
    total_fitness = sum(fitness_values)
    if total_fitness == 0:
        return random.choice(population)
    pick = random.uniform(0, total_fitness)
    current = 0
    for individu, fitness in zip(population, fitness_values):
        current += fitness
        if current > pick:
            return individu
    
    return population[-1]


print("Fungsi seleksi_roulette ")
print()
print("Ilustrasi: individu dengan fitness lebih tinggi")
print("lebih sering muncul saat dipilih berkali-kali:")
pop_kecil   = [[1]*8, [13]*8, [7,5,14,5,20,9,11,1]]  # buruk, sedang, sempurna
fit_kecil   = [calculate_fitness(p, target) for p in pop_kecil]
for p, f in zip(pop_kecil, fit_kecil):
    print(f"  {angka_ke_huruf(p)} -> fitness {f}")

Fungsi seleksi_roulette 

Ilustrasi: individu dengan fitness lebih tinggi
lebih sering muncul saat dipilih berkali-kali:
  AAAAAAAA -> fitness 150
  MMMMMMMM -> fitness 156
  GENETIKA -> fitness 200


# Crossover

menggabungkan dua parent untuk membuat anak baru.

In [19]:
def crossover(parent1, parent2, prob_co):
    anak1 = parent1.copy()
    anak2 = parent2.copy()
    
    if random.random() < prob_co:
        titik1 = random.randint(1, panjang_gen - 2)
        titik2 = random.randint(titik1 + 1, panjang_gen - 1)
        
        anak1[titik1:titik2] = parent2[titik1:titik2]
        anak2[titik1:titik2] = parent1[titik1:titik2]
        
    return anak1, anak2

individu1 = [14, 6, 2, 4, 23, 15, 17, 8]
individu2 = [12, 22, 7, 13, 11, 6, 4, 16]

anak1, anak2 = crossover(individu1, individu2, prob_co=1.0)  # paksa crossover
print("Uji Fungsi Crossover:")
print(f"  Parent 1: {individu1} -> '{angka_ke_huruf(individu1)}'")
print(f"  Parent 2: {individu2} -> '{angka_ke_huruf(individu2)}'")
print(f"  Anak 1   : {anak1} -> '{angka_ke_huruf(anak1)}'")
print(f"  Anak 2   : {anak2} -> '{angka_ke_huruf(anak2)}'")


Uji Fungsi Crossover:
  Parent 1: [14, 6, 2, 4, 23, 15, 17, 8] -> 'NFBDWOQH'
  Parent 2: [12, 22, 7, 13, 11, 6, 4, 16] -> 'LVGMKFDP'
  Anak 1   : [14, 6, 7, 13, 11, 6, 17, 8] -> 'NFGMKFQH'
  Anak 2   : [12, 22, 2, 4, 23, 15, 4, 16] -> 'LVBDWODP'


# Mutation

ubah sedikit nilai gen secara acak 

In [20]:
def mutation(individu, prob_mutasi):
   
   hasil = individu.copy()
   if random.random() < prob_mutasi:
    posisi = random.randint(0, panjang_gen - 1)
    delta = random.choice([-5,-4,-3,-2,-1,1,2,3,4,5])
    nilai_baru = hasil[posisi] + delta
    nilai_baru = max(min_allele, min(max_allele, nilai_baru))
    hasil[posisi] = nilai_baru
   return hasil


individu_sebelum =  [8, 5, 14, 11, 19, 6, 11, 1]
individu_setelah = mutation(individu_sebelum, prob_mutasi=1.0)  
print("Uji Fungsi Mutasi:")
print(f"  Sebelum: {individu_sebelum} -> '{angka_ke_huruf(individu_sebelum)}'")
print(f"  Setelah : {individu_setelah} -> '{angka_ke_huruf(individu_setelah)}'")


Uji Fungsi Mutasi:
  Sebelum: [8, 5, 14, 11, 19, 6, 11, 1] -> 'HENKSFKA'
  Setelah : [8, 5, 14, 11, 19, 6, 11, 1] -> 'HENKSFKA'


# Elitism

gabungan induk + anak, lalu diambil yang n terbaik.

In [21]:
def elitism(new_population, old_population, target, n ):
    gabungan = new_population + old_population
    
    fitness_gabungan = calculate_fitness_all(gabungan, target)
    sorted_indices = np.argsort(fitness_gabungan)[::-1]
    return [gabungan[i] for i in sorted_indices[:n]]    

print("Uji Fungsi Elitism:")
pop_lama = [[1]*8, [13]*8, [7,5,14,5,20,9,11,1]]  # buruk, sedang, sempurna
pop_baru = [[2]*8, [12]*8, [7,5,14,5,20,9,11,1]]  # sedikit lebih baik, sedikit lebih buruk, sempurna
elit = elitism(pop_baru, pop_lama, target, n=3) 
print("Populasi Lama:")
for p in pop_lama:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")
print("Populasi Baru:")
for p in pop_baru:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")
print("Populasi Elit:")
for p in elit:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")

Uji Fungsi Elitism:
Populasi Lama:
  AAAAAAAA -> fitness 150
  MMMMMMMM -> fitness 156
  GENETIKA -> fitness 200
Populasi Baru:
  BBBBBBBB -> fitness 156
  LLLLLLLL -> fitness 160
  GENETIKA -> fitness 200
Populasi Elit:
  GENETIKA -> fitness 200
  GENETIKA -> fitness 200
  LLLLLLLL -> fitness 160


# Run Algoritm

In [22]:
def run_ga():
    print("Menjalankan Algoritma Genetika untuk Word Matching:")
    print(f'target: {Target_word} -> {target}')
    
    populasi = create_population(jumlah_populasi)
    riwayat_fitness = []
    riwayat_kata = []
    
    for gen in range(jumlah_generasi):
        fitness_values = calculate_fitness_all(populasi, target)
        
        best_fitness = max(fitness_values)
        best_individu = populasi[fitness_values.index(best_fitness)]
        
        riwayat_fitness.append(best_fitness)
        riwayat_kata.append(angka_ke_huruf(best_individu))
        
        print(f'Generasi {gen+1}: Best Individu: {best_individu} -> "{angka_ke_huruf(best_individu)}" dengan Fitness: {best_fitness}')
        
        if best_fitness == max_fitness:
            print("Target tercapai!")
            break
        
        new_population = []
        
        while len(new_population) < jumlah_populasi:
            parent1 = select_roullette(populasi, fitness_values)
            parent2 = select_roullette(populasi, fitness_values)
            
            anak1, anak2 = crossover(parent1, parent2, prob_crossover)
            anak1 = mutation(anak1, prob_mutasi)
            anak2 = mutation(anak2, prob_mutasi)
            
            new_population.extend([anak1, anak2])
        
        populasi = elitism(new_population, populasi, target, jumlah_populasi)
        
    
jalankan_ga = run_ga()

Menjalankan Algoritma Genetika untuk Word Matching:
target: GENETIKA -> [6, 4, 13, 4, 19, 8, 10, 0]
Generasi 1: Best Individu: [6, 2, 20, 4, 22, 10, 13, 13] -> "FBTDVJMM" dengan Fitness: 178
Generasi 2: Best Individu: [6, 2, 20, 4, 22, 9, 13, 13] -> "FBTDVIMM" dengan Fitness: 179
Generasi 3: Best Individu: [7, 5, 17, 6, 14, 16, 15, 1] -> "GEQFNPOA" dengan Fitness: 181
Generasi 4: Best Individu: [6, 4, 16, 2, 15, 13, 9, 1] -> "FDPBOMIA" dengan Fitness: 192
Generasi 5: Best Individu: [6, 4, 16, 2, 15, 11, 11, 1] -> "FDPBOKKA" dengan Fitness: 194
Generasi 6: Best Individu: [6, 4, 16, 2, 16, 11, 11, 1] -> "FDPBPKKA" dengan Fitness: 195
Generasi 7: Best Individu: [6, 4, 16, 2, 16, 11, 11, 1] -> "FDPBPKKA" dengan Fitness: 195
Generasi 8: Best Individu: [6, 4, 16, 2, 20, 11, 11, 1] -> "FDPBTKKA" dengan Fitness: 197
Generasi 9: Best Individu: [6, 4, 16, 2, 20, 6, 11, 1] -> "FDPBTFKA" dengan Fitness: 198
Generasi 10: Best Individu: [7, 5, 17, 3, 20, 9, 10, 1] -> "GEQCTIJA" dengan Fitness: 198


# Implementasi menggunakan Library (PyGAD)

import library terlebih dahulu

In [23]:
import pygad
import numpy as np
import matplotlib.pyplot as plt

# Definisi Target Fitness dan Fitness function untuk PyGAD

In [ ]:
TARGET_WORD = "GENETIKA"
TARGET      = [ord(c) - ord('A') + 1 for c in TARGET_WORD]
MAX_ALLELE  = 26
MAX_FITNESS = MAX_ALLELE * len(TARGET)

riwayat_pygad = []

def fitness_func_pygad(ga_instance, solution, solution_idx):
    solution_int = [int(round(g)) for g in solution]
    
    total_selisih = sum(abs(solution_int[i] - TARGET[i]) for i in range(len(TARGET)))
    fitness = (len(TARGET) * MAX_ALLELE) - total_selisih
    return float(fitness)

def on_generation(ga_instance):
    best_solution, best_fitness, _ = ga_instance.best_solution()
    best_solution_int = [int(round(g)) for g in best_solution]
    kata = ''.join([chr(g + ord('A') - 1) for g in best_solution_int])
    
    riwayat_pygad.append(best_fitness)
    
    gen = ga_instance.generations_completed
    if gen == 1 or gen % 20 == 0 or best_fitness == MAX_FITNESS:
        print(f"  Gen {gen:4d} | Fitness: {best_fitness:6.1f}/{MAX_FITNESS} | Best: '{kata}'")
    
    if best_fitness >= MAX_FITNESS:
        print(f"\n  ✅ TARGET DITEMUKAN di generasi ke-{gen}!")
        return 'stop'

print("Fungsi fitness dan callback PyGAD siap ✅")

Fungsi fitness dan callback PyGAD siap ✅


# Konfigurasi & Run PyGAD

In [26]:
riwayat_pygad = []  # reset

print("=" * 60)
print("  MEMULAI GA DENGAN PYGAD")
print(f"  Target: '{TARGET_WORD}' = {TARGET}")
print("=" * 60)

ga = pygad.GA(
    # === POPULASI ===
    num_generations        = 500,          # Maksimum generasi
    sol_per_pop            = 50,           # Jumlah individu per populasi
    num_parents_mating     = 20,           # Berapa induk yang dipilih untuk reproduksi
    
    num_genes              = len(TARGET),  # Panjang kromosom = 8 (huruf)
    gene_type              = int,          # Tipe gen = integer
    init_range_low         = 1,            # Nilai minimum allele
    init_range_high        = 26,           # Nilai maksimum allele
    fitness_func           = fitness_func_pygad,
    parent_selection_type  = "rws",        # rws = Roulette Wheel Selection
    crossover_type         = "two_points", # Two-point crossover
    crossover_probability  = 0.8,
    mutation_type          = "random",     # Mutasi acak
    mutation_probability   = 0.1,
    random_mutation_min_val = -5,          # Delta mutasi min
    random_mutation_max_val =  5,          # Delta mutasi max
    keep_elitism           = 5,            # 5 individu terbaik langsung lolos
    gene_space             = range(1, 27), # Batasi allele 1-26
    on_generation          = on_generation,
    suppress_warnings      = True
)

ga.run()
print("=" * 60)

best_solution, best_fitness, _ = ga.best_solution()
best_int  = [int(round(g)) for g in best_solution]
kata_best = ''.join([chr(g + ord('A') - 1) for g in best_int])

print(f" HASIL AKHIR PyGAD:")
print(f"  Individu terbaik : {best_int}")
print(f"  Kata             : '{kata_best}'")
print(f"  Fitness          : {best_fitness}/{MAX_FITNESS}")

  MEMULAI GA DENGAN PYGAD
  Target: 'GENETIKA' = [7, 5, 14, 5, 20, 9, 11, 1]
  Gen    1 | Fitness:  169.0/208 | Best: 'WFUHTECA'
  Gen   20 | Fitness:  202.0/208 | Best: 'FFNEVIIA'
  Gen   40 | Fitness:  202.0/208 | Best: 'FFNESJIA'
  Gen   60 | Fitness:  202.0/208 | Best: 'FFNESHIA'
  Gen   80 | Fitness:  204.0/208 | Best: 'HFNESJKA'
  Gen  100 | Fitness:  204.0/208 | Best: 'FFNESHKA'
  Gen  120 | Fitness:  205.0/208 | Best: 'HFNESIKA'
  Gen  140 | Fitness:  205.0/208 | Best: 'HFNESIKA'
  Gen  160 | Fitness:  205.0/208 | Best: 'HFNESIKA'
  Gen  180 | Fitness:  207.0/208 | Best: 'HENETIKA'
  Gen  200 | Fitness:  207.0/208 | Best: 'HENETIKA'
  Gen  205 | Fitness:  208.0/208 | Best: 'GENETIKA'

  ✅ TARGET DITEMUKAN di generasi ke-205!
 HASIL AKHIR PyGAD:
  Individu terbaik : [7, 5, 14, 5, 20, 9, 11, 1]
  Kata             : 'GENETIKA'
  Fitness          : 208.0/208
